# 04 - SECOP 2025 | Índice descriptivo de prioridad

Este notebook desarrolla la **Actividad 4: Índice de prioridad**.

## Objetivo

Diseñar un índice descriptivo que permita ordenar contratos según señales de revisión prioritaria.

El índice incluye:

- Valor alto.
- Número de adiciones.
- Avance bajo.
- Modalidad contractual.
- Tema detectado.
- Texto insuficiente o ambiguo.

## Resultado esperado

Al finalizar este notebook se generan:

1. **Ranking de contratos.**
2. **Niveles de prioridad baja, media y alta.**
3. **Explicación de la fórmula.**
4. **Resumen por nivel de prioridad.**
5. **Manifiesto de trazabilidad de la Actividad 4.**

## Entrada principal

Este notebook usa como entrada preferente la salida de la Actividad 3:

```text
workspace.default.gold_secop_contratos_temas
```

Si esta tabla no existe, usa como respaldo:

```text
workspace.default.gold_secop_contratos_integrados
```

## Salidas principales

```text
workspace.default.gold_secop_indice_prioridad_contratos
workspace.default.gold_secop_ranking_prioridad
workspace.default.gold_secop_resumen_prioridad
workspace.default.qa_formula_indice_prioridad
workspace.default.qa_activity_4_summary
```

## 1. Configuración general

Se definen rutas, catálogo, esquema, tablas de entrada y tablas de salida. La ejecución está pensada para Databricks con Spark y Delta Lake.

In [0]:
from datetime import datetime, timezone
import json

from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

BASE_PATH = "/Volumes/workspace/default/tallerspark/secop"

CATALOG = "workspace"
SCHEMA = "default"

TABLE_CONTRATOS_TEMAS = f"{CATALOG}.{SCHEMA}.gold_secop_contratos_temas"
TABLE_GOLD_INTEGRADA = f"{CATALOG}.{SCHEMA}.gold_secop_contratos_integrados"

TABLE_INDICE_PRIORIDAD = f"{CATALOG}.{SCHEMA}.gold_secop_indice_prioridad_contratos"
TABLE_RANKING_PRIORIDAD = f"{CATALOG}.{SCHEMA}.gold_secop_ranking_prioridad"
TABLE_RESUMEN_PRIORIDAD = f"{CATALOG}.{SCHEMA}.gold_secop_resumen_prioridad"
TABLE_FORMULA_PRIORIDAD = f"{CATALOG}.{SCHEMA}.qa_formula_indice_prioridad"
TABLE_ACTIVITY_4_SUMMARY = f"{CATALOG}.{SCHEMA}.qa_activity_4_summary"

OUTPUT_PATH_GOLD = f"{BASE_PATH}/gold/secop_indice_prioridad"
OUTPUT_PATH_MANIFEST = f"{BASE_PATH}/manifest"

print("BASE_PATH:", BASE_PATH)
print("Entrada preferida:", TABLE_CONTRATOS_TEMAS)
print("Salida índice:", TABLE_INDICE_PRIORIDAD)
print("Salida ranking:", TABLE_RANKING_PRIORIDAD)

## 2. Lectura de la tabla base

El notebook intenta leer primero la tabla de contratos con temas de la Actividad 3. Esta es la mejor entrada porque ya contiene `texto_busqueda` y `temas_detectados`.

Si no existe, se usa la tabla integrada de la Actividad 2 como respaldo.

In [0]:
def table_exists(table_name):
    try:
        spark.table(table_name).limit(1).count()
        return True
    except Exception:
        return False

if table_exists(TABLE_CONTRATOS_TEMAS):
    df_base = spark.table(TABLE_CONTRATOS_TEMAS)
    INPUT_TABLE_USED = TABLE_CONTRATOS_TEMAS
    INPUT_LAYER = "activity_3_topics"
elif table_exists(TABLE_GOLD_INTEGRADA):
    df_base = spark.table(TABLE_GOLD_INTEGRADA)
    INPUT_TABLE_USED = TABLE_GOLD_INTEGRADA
    INPUT_LAYER = "activity_2_gold_fallback"
else:
    raise ValueError("No existe la tabla de Actividad 3 ni la tabla Gold integrada de Actividad 2.")

print("Tabla usada:", INPUT_TABLE_USED)
print("Capa usada:", INPUT_LAYER)
print("Filas:", df_base.count())
print("Columnas:", len(df_base.columns))

display(df_base.limit(5))

## 3. Preparación defensiva de columnas

El índice necesita variables mínimas. Si alguna columna no existe, se crea con un valor neutral para que el notebook pueda ejecutarse sin romperse.

Variables usadas:

- `id_contrato_std`
- `valor_contrato_num`
- `numero_adiciones`
- `ultimo_avance_real_num`
- `modalidad_contratacion_std`
- `temas_detectados`
- `texto_busqueda`
- `longitud_texto_busqueda`

In [0]:
df_work = df_base

if "id_contrato_std" not in df_work.columns:
    df_work = df_work.withColumn("id_contrato_std", F.monotonically_increasing_id().cast("string"))

if "valor_contrato_num" not in df_work.columns:
    df_work = df_work.withColumn("valor_contrato_num", F.lit(None).cast("double"))

if "numero_adiciones" not in df_work.columns:
    df_work = df_work.withColumn("numero_adiciones", F.lit(0).cast("int"))

if "ultimo_avance_real_num" not in df_work.columns:
    if "avance_real_num" in df_work.columns:
        df_work = df_work.withColumn("ultimo_avance_real_num", F.col("avance_real_num").cast("double"))
    else:
        df_work = df_work.withColumn("ultimo_avance_real_num", F.lit(None).cast("double"))

if "modalidad_contratacion_std" not in df_work.columns:
    df_work = df_work.withColumn("modalidad_contratacion_std", F.lit(None).cast("string"))

if "texto_busqueda" not in df_work.columns:
    text_candidates = [c for c in ["objeto_contrato_std", "nombre_entidad_std", "proveedor_std", "sector_std", "tipo_contrato_std", "estado_contrato_std"] if c in df_work.columns]
    if text_candidates:
        df_work = df_work.withColumn("texto_busqueda", F.lower(F.concat_ws(" ", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in text_candidates])))
    else:
        df_work = df_work.withColumn("texto_busqueda", F.lit("").cast("string"))

if "longitud_texto_busqueda" not in df_work.columns:
    df_work = df_work.withColumn("longitud_texto_busqueda", F.length(F.col("texto_busqueda")))

if "temas_detectados" not in df_work.columns:
    df_work = df_work.withColumn("temas_detectados", F.array(F.lit("sin_tema_detectado")))

df_work = (
    df_work
    .withColumn("valor_contrato_num", F.col("valor_contrato_num").cast("double"))
    .withColumn("numero_adiciones", F.coalesce(F.col("numero_adiciones").cast("int"), F.lit(0)))
    .withColumn("ultimo_avance_real_num", F.col("ultimo_avance_real_num").cast("double"))
    .withColumn("modalidad_contratacion_std", F.upper(F.coalesce(F.col("modalidad_contratacion_std").cast("string"), F.lit(""))))
    .withColumn("texto_busqueda", F.coalesce(F.col("texto_busqueda").cast("string"), F.lit("")))
    .withColumn("longitud_texto_busqueda", F.coalesce(F.col("longitud_texto_busqueda").cast("int"), F.lit(0)))
)

print("Columnas mínimas listas para calcular el índice.")

## 4. Cálculo de umbrales de valor alto

El puntaje de valor contractual se calcula usando percentiles del propio dataset. Esto evita usar umbrales arbitrarios.

Se calculan:

- P50: mediana.
- P75: contratos de valor superior al 75 % de la distribución.
- P90: contratos de valor alto.
- P95: contratos de valor muy alto.

In [0]:
df_values_non_null = df_work.filter(F.col("valor_contrato_num").isNotNull() & (F.col("valor_contrato_num") > 0))

if df_values_non_null.limit(1).count() > 0:
    p50, p75, p90, p95 = df_values_non_null.approxQuantile("valor_contrato_num", [0.50, 0.75, 0.90, 0.95], 0.01)
else:
    p50, p75, p90, p95 = 0.0, 0.0, 0.0, 0.0

VALUE_P50 = float(p50)
VALUE_P75 = float(p75)
VALUE_P90 = float(p90)
VALUE_P95 = float(p95)

print("P50:", VALUE_P50)
print("P75:", VALUE_P75)
print("P90:", VALUE_P90)
print("P95:", VALUE_P95)

## 5. Fórmula del índice de prioridad

El índice usa una escala de **0 a 100 puntos**.

```text
indice_prioridad =
    score_valor_alto
  + score_adiciones
  + score_avance_bajo
  + score_modalidad
  + score_tema
  + score_texto
```

Distribución:

| Componente | Máximo |
|---|---:|
| Valor alto | 25 |
| Número de adiciones | 20 |
| Avance bajo | 20 |
| Modalidad contractual | 15 |
| Tema detectado | 10 |
| Texto insuficiente o ambiguo | 10 |

Niveles:

```text
BAJA  = índice < 40
MEDIA = 40 <= índice < 70
ALTA  = índice >= 70
```

Este índice **no declara irregularidades**. Solo ordena contratos para priorizar revisión descriptiva.

In [0]:
formula_rows = [
    {"componente":"score_valor_alto","puntaje_maximo":25,"criterio":"Mayor puntaje para contratos en percentiles altos de valor.","regla":f">=P95:25; >=P90:20; >=P75:15; >=P50:8; menor a P50:3; nulo o <=0:0. P50={VALUE_P50}, P75={VALUE_P75}, P90={VALUE_P90}, P95={VALUE_P95}.","limitacion":"El umbral depende de la distribución del dataset."},
    {"componente":"score_adiciones","puntaje_maximo":20,"criterio":"Mayor puntaje si el contrato tiene varias adiciones.","regla":">=5 adiciones:20; >=3:15; >=1:8; 0:0.","limitacion":"Una adición puede ser normal; solo prioriza revisión."},
    {"componente":"score_avance_bajo","puntaje_maximo":20,"criterio":"Mayor puntaje si el avance es bajo o no existe información.","regla":"avance nulo:5; <25%:20; <50%:15; <80%:8; >=80%:0.","limitacion":"El avance bajo puede depender de la etapa del contrato."},
    {"componente":"score_modalidad","puntaje_maximo":15,"criterio":"Puntaje según modalidad contractual.","regla":"Contratación directa:15; régimen especial/selección abreviada/mínima cuantía:10; licitación/concurso:5; otras:3.","limitacion":"La modalidad no implica irregularidad."},
    {"componente":"score_tema","puntaje_maximo":10,"criterio":"Puntaje según tema detectado.","regla":"Temas estratégicos:10; temas medios:6; otros:3; sin tema:0.","limitacion":"Depende de reglas de palabras clave."},
    {"componente":"score_texto","puntaje_maximo":10,"criterio":"Puntaje por texto corto, vacío o ambiguo.","regla":"vacío o sin tema:10; longitud <50:10; <150:6; <300:3; >=300:0.","limitacion":"Texto corto no necesariamente implica mala calidad contractual."}
]

df_formula = spark.createDataFrame(formula_rows)
display(df_formula)

(df_formula.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE_FORMULA_PRIORIDAD))
print("Tabla de fórmula guardada:", TABLE_FORMULA_PRIORIDAD)

## 6. Cálculo de puntajes parciales

Cada componente se calcula como una columna independiente. Esto permite auditar por qué un contrato obtiene un nivel bajo, medio o alto.

In [0]:
HIGH_PRIORITY_TOPICS = ["infraestructura_obras", "salud", "tecnologia_software", "seguridad_vigilancia", "servicios_profesionales", "interventoria_supervision", "juridico_administrativo"]
MEDIUM_PRIORITY_TOPICS = ["mantenimiento", "suministros_compras", "transporte", "ambiente", "alimentacion", "educacion"]

high_topics_array_sql = "array(" + ",".join([f"'{x}'" for x in HIGH_PRIORITY_TOPICS]) + ")"
medium_topics_array_sql = "array(" + ",".join([f"'{x}'" for x in MEDIUM_PRIORITY_TOPICS]) + ")"

df_scores = (
    df_work
    .withColumn("score_valor_alto", F.when(F.col("valor_contrato_num").isNull() | (F.col("valor_contrato_num") <= 0), F.lit(0)).when(F.col("valor_contrato_num") >= F.lit(VALUE_P95), F.lit(25)).when(F.col("valor_contrato_num") >= F.lit(VALUE_P90), F.lit(20)).when(F.col("valor_contrato_num") >= F.lit(VALUE_P75), F.lit(15)).when(F.col("valor_contrato_num") >= F.lit(VALUE_P50), F.lit(8)).otherwise(F.lit(3)))
    .withColumn("score_adiciones", F.when(F.col("numero_adiciones") >= 5, F.lit(20)).when(F.col("numero_adiciones") >= 3, F.lit(15)).when(F.col("numero_adiciones") >= 1, F.lit(8)).otherwise(F.lit(0)))
    .withColumn("score_avance_bajo", F.when(F.col("ultimo_avance_real_num").isNull(), F.lit(5)).when(F.col("ultimo_avance_real_num") < 25, F.lit(20)).when(F.col("ultimo_avance_real_num") < 50, F.lit(15)).when(F.col("ultimo_avance_real_num") < 80, F.lit(8)).otherwise(F.lit(0)))
    .withColumn("score_modalidad", F.when(F.col("modalidad_contratacion_std").contains("CONTRATACION DIRECTA") | F.col("modalidad_contratacion_std").contains("CONTRATACIÓN DIRECTA"), F.lit(15)).when(F.col("modalidad_contratacion_std").contains("REGIMEN ESPECIAL") | F.col("modalidad_contratacion_std").contains("RÉGIMEN ESPECIAL") | F.col("modalidad_contratacion_std").contains("SELECCION ABREVIADA") | F.col("modalidad_contratacion_std").contains("SELECCIÓN ABREVIADA") | F.col("modalidad_contratacion_std").contains("MINIMA CUANTIA") | F.col("modalidad_contratacion_std").contains("MÍNIMA CUANTÍA"), F.lit(10)).when(F.col("modalidad_contratacion_std").contains("LICITACION PUBLICA") | F.col("modalidad_contratacion_std").contains("LICITACIÓN PÚBLICA") | F.col("modalidad_contratacion_std").contains("CONCURSO DE MERITOS") | F.col("modalidad_contratacion_std").contains("CONCURSO DE MÉRITOS"), F.lit(5)).otherwise(F.lit(3)))
    .withColumn("score_tema", F.when(F.expr(f"size(array_intersect(temas_detectados, {high_topics_array_sql})) > 0"), F.lit(10)).when(F.expr(f"size(array_intersect(temas_detectados, {medium_topics_array_sql})) > 0"), F.lit(6)).when(F.array_contains(F.col("temas_detectados"), "sin_tema_detectado"), F.lit(0)).otherwise(F.lit(3)))
    .withColumn("score_texto", F.when(F.col("texto_busqueda").isNull() | (F.trim(F.col("texto_busqueda")) == ""), F.lit(10)).when(F.array_contains(F.col("temas_detectados"), "sin_tema_detectado"), F.lit(10)).when(F.col("longitud_texto_busqueda") < 50, F.lit(10)).when(F.col("longitud_texto_busqueda") < 150, F.lit(6)).when(F.col("longitud_texto_busqueda") < 300, F.lit(3)).otherwise(F.lit(0)))
)

display(df_scores.select("id_contrato_std", "valor_contrato_num", "numero_adiciones", "ultimo_avance_real_num", "modalidad_contratacion_std", "temas_detectados", "longitud_texto_busqueda", "score_valor_alto", "score_adiciones", "score_avance_bajo", "score_modalidad", "score_tema", "score_texto").limit(20))

## 7. Índice final y niveles de prioridad

Se suman los puntajes parciales para obtener `indice_prioridad`. Luego se asigna el nivel:

- `ALTA`
- `MEDIA`
- `BAJA`

También se crea `factores_prioridad`, que explica qué señales elevaron el puntaje.

In [0]:
df_indice = (
    df_scores
    .withColumn("indice_prioridad", (F.col("score_valor_alto") + F.col("score_adiciones") + F.col("score_avance_bajo") + F.col("score_modalidad") + F.col("score_tema") + F.col("score_texto")).cast("double"))
    .withColumn("nivel_prioridad", F.when(F.col("indice_prioridad") >= 70, F.lit("ALTA")).when(F.col("indice_prioridad") >= 40, F.lit("MEDIA")).otherwise(F.lit("BAJA")))
    .withColumn("factor_valor_alto", F.when(F.col("score_valor_alto") >= 15, F.lit("valor alto")))
    .withColumn("factor_adiciones", F.when(F.col("score_adiciones") >= 8, F.lit("contrato con adiciones")))
    .withColumn("factor_avance", F.when(F.col("score_avance_bajo") >= 15, F.lit("avance bajo")))
    .withColumn("factor_modalidad", F.when(F.col("score_modalidad") >= 10, F.lit("modalidad contractual priorizada")))
    .withColumn("factor_tema", F.when(F.col("score_tema") >= 10, F.lit("tema sensible o estratégico")))
    .withColumn("factor_texto", F.when(F.col("score_texto") >= 6, F.lit("texto insuficiente o ambiguo")))
    .withColumn("factores_prioridad", F.array_remove(F.array("factor_valor_alto", "factor_adiciones", "factor_avance", "factor_modalidad", "factor_tema", "factor_texto"), None))
    .withColumn("explicacion_indice", F.concat(F.lit("Índice descriptivo calculado sobre 100 puntos. Nivel: "), F.col("nivel_prioridad"), F.lit(". Factores principales: "), F.concat_ws(", ", F.col("factores_prioridad")), F.lit(".")))
    .drop("factor_valor_alto", "factor_adiciones", "factor_avance", "factor_modalidad", "factor_tema", "factor_texto")
    .withColumn("_activity_4_created_at", F.current_timestamp())
)

print("Contratos con índice:", df_indice.count())
display(df_indice.select("id_contrato_std", "indice_prioridad", "nivel_prioridad", "factores_prioridad", "score_valor_alto", "score_adiciones", "score_avance_bajo", "score_modalidad", "score_tema", "score_texto", "explicacion_indice").limit(20))

## 8. Ranking de contratos

El ranking se ordena por:

1. Mayor índice de prioridad.
2. Mayor valor contractual.
3. Mayor número de adiciones.

Este resultado cumple el requisito de **ranking de contratos**.

In [0]:
ranking_window = Window.orderBy(F.col("indice_prioridad").desc(), F.col("valor_contrato_num").desc_nulls_last(), F.col("numero_adiciones").desc_nulls_last())

df_ranking = df_indice.withColumn("ranking_prioridad", F.row_number().over(ranking_window))

preferred_columns = ["ranking_prioridad", "id_contrato_std", "indice_prioridad", "nivel_prioridad", "factores_prioridad", "valor_contrato_num", "numero_adiciones", "ultimo_avance_real_num", "modalidad_contratacion_std", "temas_detectados", "longitud_texto_busqueda", "nombre_entidad_std", "proveedor_std", "departamento_std", "ciudad_std", "objeto_contrato_std", "texto_busqueda", "score_valor_alto", "score_adiciones", "score_avance_bajo", "score_modalidad", "score_tema", "score_texto", "explicacion_indice"]
selected_columns = [c for c in preferred_columns if c in df_ranking.columns]

df_ranking_final = df_ranking.select(*selected_columns)
display(df_ranking_final.limit(50))

## 9. Resumen por nivel de prioridad

Se calcula la cantidad de contratos por nivel y métricas agregadas de valor, índice, adiciones y avance.

In [0]:
summary_aggs = [
    F.countDistinct("id_contrato_std").alias("contratos_distintos"),
    F.count("*").alias("registros"),
    F.avg("indice_prioridad").alias("indice_promedio"),
    F.min("indice_prioridad").alias("indice_minimo"),
    F.max("indice_prioridad").alias("indice_maximo"),
    F.avg("numero_adiciones").alias("promedio_adiciones"),
    F.avg("ultimo_avance_real_num").alias("promedio_ultimo_avance"),
    F.sum("valor_contrato_num").alias("valor_total_contratos"),
    F.avg("valor_contrato_num").alias("valor_promedio_contrato"),
    F.max("valor_contrato_num").alias("valor_maximo_contrato")
]

total_contracts = df_indice.select("id_contrato_std").distinct().count()

df_resumen_prioridad = (
    df_indice
    .groupBy("nivel_prioridad")
    .agg(*summary_aggs)
    .withColumn("porcentaje_contratos", F.round((F.col("contratos_distintos") / F.lit(total_contracts)) * 100, 4))
    .orderBy(F.when(F.col("nivel_prioridad") == "ALTA", 1).when(F.col("nivel_prioridad") == "MEDIA", 2).otherwise(3))
)

display(df_resumen_prioridad)

## 10. Resumen por tema y nivel de prioridad

Este cruce permite identificar qué temas concentran contratos con prioridad alta, media o baja.

In [0]:
if "temas_detectados" in df_indice.columns:
    df_tema_nivel = (
        df_indice
        .withColumn("tema_detectado", F.explode(F.col("temas_detectados")))
        .groupBy("tema_detectado", "nivel_prioridad")
        .agg(F.countDistinct("id_contrato_std").alias("contratos_distintos"), F.avg("indice_prioridad").alias("indice_promedio"), F.sum("valor_contrato_num").alias("valor_total_contratos"))
        .orderBy(F.desc("contratos_distintos"))
    )
    display(df_tema_nivel)
else:
    print("No existe columna temas_detectados.")

## 11. Escritura de tablas Delta

Se guardan las salidas principales en Unity Catalog.

In [0]:
(df_indice.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE_INDICE_PRIORIDAD))
(df_ranking_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE_RANKING_PRIORIDAD))
(df_resumen_prioridad.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE_RESUMEN_PRIORIDAD))

print("Tabla índice guardada:", TABLE_INDICE_PRIORIDAD)
print("Tabla ranking guardada:", TABLE_RANKING_PRIORIDAD)
print("Tabla resumen guardada:", TABLE_RESUMEN_PRIORIDAD)

## 12. Escritura de copias Parquet

Además de Delta, se guardan copias Parquet en el Volume del proyecto.

In [0]:
indice_output_path = f"{OUTPUT_PATH_GOLD}/indice_prioridad_contratos"
ranking_output_path = f"{OUTPUT_PATH_GOLD}/ranking_prioridad"
resumen_output_path = f"{OUTPUT_PATH_GOLD}/resumen_prioridad"
formula_output_path = f"{OUTPUT_PATH_GOLD}/formula_prioridad"

df_indice.write.mode("overwrite").parquet(indice_output_path)
df_ranking_final.write.mode("overwrite").parquet(ranking_output_path)
df_resumen_prioridad.write.mode("overwrite").parquet(resumen_output_path)
df_formula.write.mode("overwrite").parquet(formula_output_path)

print("Parquet índice:", indice_output_path)
print("Parquet ranking:", ranking_output_path)
print("Parquet resumen:", resumen_output_path)
print("Parquet fórmula:", formula_output_path)

## 13. Resumen de cumplimiento de la Actividad 4

Esta tabla evidencia el cumplimiento de los resultados solicitados.

In [0]:
activity_4_summary = [
    {"requirement":"Diseñar índice descriptivo", "result":"Índice de prioridad creado sobre escala de 0 a 100 puntos.", "output":TABLE_INDICE_PRIORIDAD, "status":"OK"},
    {"requirement":"Incluir valor alto", "result":"score_valor_alto calculado con percentiles P50, P75, P90 y P95.", "output":TABLE_FORMULA_PRIORIDAD, "status":"OK"},
    {"requirement":"Incluir número de adiciones", "result":"score_adiciones calculado con base en cantidad de adiciones por contrato.", "output":TABLE_INDICE_PRIORIDAD, "status":"OK"},
    {"requirement":"Incluir avance bajo", "result":"score_avance_bajo calculado con base en el último avance disponible.", "output":TABLE_INDICE_PRIORIDAD, "status":"OK"},
    {"requirement":"Incluir modalidad contractual", "result":"score_modalidad calculado con reglas descriptivas por modalidad.", "output":TABLE_INDICE_PRIORIDAD, "status":"OK"},
    {"requirement":"Incluir tema detectado", "result":"score_tema calculado usando temas_detectados de la Actividad 3.", "output":TABLE_INDICE_PRIORIDAD, "status":"OK"},
    {"requirement":"Incluir texto insuficiente o ambiguo", "result":"score_texto calculado con longitud de texto_busqueda y sin_tema_detectado.", "output":TABLE_INDICE_PRIORIDAD, "status":"OK"},
    {"requirement":"Ranking de contratos", "result":"Ranking generado por índice, valor y número de adiciones.", "output":TABLE_RANKING_PRIORIDAD, "status":"OK"},
    {"requirement":"Niveles baja, media y alta", "result":"Nivel creado con rangos BAJA, MEDIA y ALTA.", "output":TABLE_RESUMEN_PRIORIDAD, "status":"OK"},
    {"requirement":"Explicación de la fórmula", "result":"Tabla qa_formula_indice_prioridad creada con reglas, puntajes y limitaciones.", "output":TABLE_FORMULA_PRIORIDAD, "status":"OK"}
]

df_activity_4_summary = spark.createDataFrame(activity_4_summary)
display(df_activity_4_summary)

(df_activity_4_summary.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE_ACTIVITY_4_SUMMARY))
print("Tabla guardada:", TABLE_ACTIVITY_4_SUMMARY)

## 14. Manifiesto final de la Actividad 4

Se guarda un manifiesto JSON con trazabilidad de la ejecución, tablas creadas, umbrales y limitaciones.

In [0]:
priority_counts = [row.asDict() for row in df_resumen_prioridad.collect()]

activity_4_manifest = {
    "project": "SECOP Big Data 2025",
    "activity": "Actividad 4 - Índice de prioridad",
    "input_table_used": INPUT_TABLE_USED,
    "input_layer": INPUT_LAYER,
    "output_tables": {
        "indice_prioridad": TABLE_INDICE_PRIORIDAD,
        "ranking_prioridad": TABLE_RANKING_PRIORIDAD,
        "resumen_prioridad": TABLE_RESUMEN_PRIORIDAD,
        "formula_prioridad": TABLE_FORMULA_PRIORIDAD,
        "activity_4_summary": TABLE_ACTIVITY_4_SUMMARY
    },
    "output_paths": {
        "indice_prioridad_parquet": indice_output_path,
        "ranking_prioridad_parquet": ranking_output_path,
        "resumen_prioridad_parquet": resumen_output_path,
        "formula_prioridad_parquet": formula_output_path
    },
    "value_thresholds": {"p50": VALUE_P50, "p75": VALUE_P75, "p90": VALUE_P90, "p95": VALUE_P95},
    "priority_levels": {"BAJA": "indice_prioridad < 40", "MEDIA": "40 <= indice_prioridad < 70", "ALTA": "indice_prioridad >= 70"},
    "priority_counts": priority_counts,
    "main_limitations": [
        "El índice es descriptivo y no implica irregularidad contractual.",
        "Los pesos fueron definidos para priorización académica y pueden ajustarse.",
        "El puntaje por temas depende de reglas de texto no estructurado.",
        "Un avance bajo puede estar asociado a la etapa natural del contrato o a datos incompletos.",
        "La modalidad contractual se usa como variable descriptiva, no como señal de incumplimiento.",
        "El texto insuficiente indica baja capacidad de interpretación, no necesariamente mala calidad contractual."
    ],
    "created_at_utc": datetime.now(timezone.utc).isoformat()
}

manifest_path = f"{OUTPUT_PATH_MANIFEST}/activity_4_priority_index_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(activity_4_manifest, f, ensure_ascii=False, indent=4)

print("Manifest saved:", manifest_path)

## 15. Texto sugerido para el informe

En la Actividad 4 se diseñó un índice descriptivo de prioridad contractual con escala de 0 a 100 puntos. El índice combina seis componentes: valor contractual alto, número de adiciones, avance bajo, modalidad contractual, tema detectado y texto insuficiente o ambiguo. Para el componente de valor se utilizaron percentiles de la distribución de contratos, mientras que los demás componentes se calcularon con reglas descriptivas y auditables.

El resultado permite ordenar los contratos en un ranking y clasificarlos en tres niveles: baja, media y alta prioridad. Esta clasificación no implica hallazgos de irregularidad, sino que funciona como una herramienta de priorización para enfocar la revisión analítica sobre contratos con mayor concentración de señales descriptivas. Adicionalmente, se generó una tabla explicativa de la fórmula, un resumen por nivel de prioridad y un manifiesto de trazabilidad con los criterios y limitaciones del método.